# Data Preparation Notebook 2: Create a CSV to train embedding model
- Create a CSV file with `variable_para` and `alternate_variable_para` 
- use semantically interoperable variable names, definitions and values from `definitions.ncit.csv` and `synonyms.ncit.csv` files

In [1]:
# imports
import pandas as pd
import numpy as np
from itertools import combinations
from itertools import zip_longest

### Read definitions

In [2]:
definitions_df = pd.read_csv('definitions.ncit.csv', index_col='concept')

In [3]:
definitions_df.head(n=3)

,definition,source
concept,,
C1000,"A recombinant form of amphiregulin, a member o...",NCI
C100000,A percutaneous coronary intervention is necess...,CDISC
C100000,A percutaneous coronary intervention is necess...,NCI


In [4]:
definitions_df.shape

(240912, 2)

### 181270 unique concepts combed through NCIt APIs 
- definitions obtained for these
- 212234 unique concepts exist in NCIt file

In [5]:
definitions_df.index.nunique()

181270

In [6]:
181270/212234

0.854104431900638

### Read synonyms

In [7]:
synonyms_df = pd.read_csv('synonyms.ncit.csv', index_col='concept')

In [8]:
synonyms_df.head(n=3)

,name,term_type,source
concept,,,
C1000,AMPHIREGULIN,PT,FDA
C1000,AR,AB,NCI
C1000,CRDGF,AB,NCI


In [9]:
synonyms_df.shape

(1064585, 3)

### How many definitions per concept are there?

In [10]:
definition_count = definitions_df.groupby('concept')['definition'].agg('count')

In [11]:
definition_count.index

Index(['C1000', 'C100000', 'C100001', 'C100002', 'C100003', 'C100004',
       'C100005', 'C100006', 'C100007', 'C100008',
       ...
       'C99990', 'C99991', 'C99992', 'C99993', 'C99994', 'C99995', 'C99996',
       'C99997', 'C99998', 'C99999'],
      dtype='object', name='concept', length=181270)

In [12]:
definition_count.head(n=3)

concept
C1000      1
C100000    2
C100001    2
Name: definition, dtype: int64

In [13]:
definition_count.value_counts()

definition
1    130219
2     44110
3      5569
4      1180
5       135
6        37
7        13
8         5
9         2
Name: count, dtype: int64

In [14]:
definitions_df.loc[definition_count == 1].shape

(130219, 2)

In [15]:
definitions_df.loc[definition_count > 1].shape

(110693, 2)

In [16]:
multiple_definition_df = definitions_df.loc[definition_count > 1]

In [17]:
multiple_definition_df.shape

(110693, 2)

In [18]:
multiple_definition_df.head(n=3)

,definition,source
concept,,
C100000,A percutaneous coronary intervention is necess...,CDISC
C100000,A percutaneous coronary intervention is necess...,NCI
C100001,A percutaneous coronary intervention is necess...,CDISC


## Group synonyms by concept to add variable names 
- for each concept, get the list of variable names
- add to above df

In [19]:
syn_by_concept = synonyms_df.groupby(level=0)['name'].agg(list)

In [20]:
syn_by_concept.head(n=3)

concept
C1000      [AMPHIREGULIN, AR, CRDGF, KAF, Recombinant Amp...
C10000     [CD(P)TH, CTX/DBD/FXM/PRED/TMX, Cyclophosphami...
C100000    [PERCUTANEOUS CORONARY INTERVENTION (PCI) FOR ...
Name: name, dtype: object

In [21]:
syn_by_concept.name = 'variable_name'

In [22]:
syn_by_concept.shape

(212228,)

### Join variable names list with multiple_definition_df

In [23]:
multiple_definitions_with_varnames = pd.merge(
    left=multiple_definition_df,
    right=syn_by_concept,
    left_index=True,
    right_index=True,
    how='inner'
)

In [24]:
multiple_definitions_with_varnames.shape

(110693, 3)

In [25]:
multiple_definitions_with_varnames.head(n=3)

,definition,source,variable_name
concept,,,
C100000,A percutaneous coronary intervention is necess...,CDISC,[PERCUTANEOUS CORONARY INTERVENTION (PCI) FOR ...
C100000,A percutaneous coronary intervention is necess...,NCI,[PERCUTANEOUS CORONARY INTERVENTION (PCI) FOR ...
C100001,A percutaneous coronary intervention is necess...,CDISC,[PERCUTANEOUS CORONARY INTERVENTION (PCI) FOR ...


### Explode variable_name column
- we want to have only one variable_name per row

In [26]:
multiple_definitions_with_varnames = multiple_definitions_with_varnames.reset_index().explode("variable_name")

In [27]:
multiple_definitions_with_varnames.head(n=5)

,concept,definition,source,variable_name
0,C100000,A percutaneous coronary intervention is necess...,CDISC,PERCUTANEOUS CORONARY INTERVENTION (PCI) FOR S...
0,C100000,A percutaneous coronary intervention is necess...,CDISC,Percutaneous Coronary Intervention for ST Elev...
0,C100000,A percutaneous coronary intervention is necess...,CDISC,Percutaneous Coronary Intervention for ST Elev...
1,C100000,A percutaneous coronary intervention is necess...,NCI,PERCUTANEOUS CORONARY INTERVENTION (PCI) FOR S...
1,C100000,A percutaneous coronary intervention is necess...,NCI,Percutaneous Coronary Intervention for ST Elev...


In [28]:
multiple_definitions_with_varnames.shape

(633375, 4)

In [29]:
multiple_definitions_with_varnames = multiple_definitions_with_varnames.set_index('concept')

### Out of 180K unique concepts how many have multiple definitions
- only 51K

In [30]:
multiple_definitions_with_varnames.index.nunique()

51051

### Create semantically interoperable definitions for the 51K concepts
- group by concept
- for each concept group, iterate through the list and make unique pairs of concept, definition, source and variable_name


In [31]:
rows = []
# group by concept
for concept, group in multiple_definitions_with_varnames.groupby(level=0):
    # convert group to list
    group = list(group.iterrows())
    # iterate through list and make pairs
    for i in range(len(group)):
        for j in range(i + 1, len(group)):
            # first row
            r1 = group[i][1]
            # second row
            r2 = group[j][1]
            rows.append({
                "concept": concept,
                "definition_1": r1["definition"],
                "source_1": r1["source"],
                "variable_name_1": r1["variable_name"],
                "definition_2": r2["definition"],
                "source_2": r2["source"],
                "variable_name_2": r2["variable_name"],
            })
unique_pairs = pd.DataFrame(rows)

In [32]:
unique_pairs.head()

,concept,definition_1,source_1,variable_name_1,definition_2,source_2,variable_name_2
0,C100000,A percutaneous coronary intervention is necess...,CDISC,PERCUTANEOUS CORONARY INTERVENTION (PCI) FOR S...,A percutaneous coronary intervention is necess...,CDISC,Percutaneous Coronary Intervention for ST Elev...
1,C100000,A percutaneous coronary intervention is necess...,CDISC,PERCUTANEOUS CORONARY INTERVENTION (PCI) FOR S...,A percutaneous coronary intervention is necess...,CDISC,Percutaneous Coronary Intervention for ST Elev...
2,C100000,A percutaneous coronary intervention is necess...,CDISC,PERCUTANEOUS CORONARY INTERVENTION (PCI) FOR S...,A percutaneous coronary intervention is necess...,NCI,PERCUTANEOUS CORONARY INTERVENTION (PCI) FOR S...
3,C100000,A percutaneous coronary intervention is necess...,CDISC,PERCUTANEOUS CORONARY INTERVENTION (PCI) FOR S...,A percutaneous coronary intervention is necess...,NCI,Percutaneous Coronary Intervention for ST Elev...
4,C100000,A percutaneous coronary intervention is necess...,CDISC,PERCUTANEOUS CORONARY INTERVENTION (PCI) FOR S...,A percutaneous coronary intervention is necess...,NCI,Percutaneous Coronary Intervention for ST Elev...


In [33]:
unique_pairs.shape

(5978218, 7)

In [34]:
# drop duplicate rows
unique_pairs_nodup = unique_pairs.drop_duplicates()

In [35]:
unique_pairs_nodup.head(n=3)

,concept,definition_1,source_1,variable_name_1,definition_2,source_2,variable_name_2
0,C100000,A percutaneous coronary intervention is necess...,CDISC,PERCUTANEOUS CORONARY INTERVENTION (PCI) FOR S...,A percutaneous coronary intervention is necess...,CDISC,Percutaneous Coronary Intervention for ST Elev...
2,C100000,A percutaneous coronary intervention is necess...,CDISC,PERCUTANEOUS CORONARY INTERVENTION (PCI) FOR S...,A percutaneous coronary intervention is necess...,NCI,PERCUTANEOUS CORONARY INTERVENTION (PCI) FOR S...
3,C100000,A percutaneous coronary intervention is necess...,CDISC,PERCUTANEOUS CORONARY INTERVENTION (PCI) FOR S...,A percutaneous coronary intervention is necess...,NCI,Percutaneous Coronary Intervention for ST Elev...


In [36]:
unique_pairs_nodup.shape

(2424371, 7)

### remove rows that have same info, but different case

In [37]:
mask = (
    unique_pairs_nodup["variable_name_1"].str.casefold().eq(unique_pairs_nodup["variable_name_2"].str.casefold()) &
    unique_pairs_nodup["definition_1"].str.casefold().eq(unique_pairs_nodup["definition_2"].str.casefold())
)

unique_pairs_nodup = unique_pairs_nodup[~mask]

In [38]:
unique_pairs_nodup.shape

(2122106, 7)

### check rows where:
- definition_1, variable_name_1 == definition_2, variable_name_2
- filter them out

In [39]:
unique_pairs_nodup_filtered = unique_pairs_nodup[
    (unique_pairs_nodup['definition_1'] != unique_pairs_nodup['definition_2']) & (unique_pairs_nodup['variable_name_1'] != unique_pairs_nodup['variable_name_2'])
]

In [40]:
unique_pairs_nodup_filtered.shape

(950827, 7)

In [41]:
unique_pairs_nodup_filtered.head(n=3)

,concept,definition_1,source_1,variable_name_1,definition_2,source_2,variable_name_2
3,C100000,A percutaneous coronary intervention is necess...,CDISC,PERCUTANEOUS CORONARY INTERVENTION (PCI) FOR S...,A percutaneous coronary intervention is necess...,NCI,Percutaneous Coronary Intervention for ST Elev...
6,C100000,A percutaneous coronary intervention is necess...,CDISC,Percutaneous Coronary Intervention for ST Elev...,A percutaneous coronary intervention is necess...,NCI,PERCUTANEOUS CORONARY INTERVENTION (PCI) FOR S...
18,C100001,A percutaneous coronary intervention is necess...,CDISC,PERCUTANEOUS CORONARY INTERVENTION (PCI) FOR S...,A percutaneous coronary intervention is necess...,NCI,Percutaneous Coronary Intervention for ST Elev...


### for each concept, further remove redundancies
- only keep rows where variable_name_1, definition_1 and variable_name_2, definition_2 are truly unique
- i.e in example below, we want to remove second row
```
concept,def1,name1,def2,name2
X,a,b,c,d
X,c,d,b,a
```

In [42]:
def remove_redundancies(df):
    pairs = df.apply(
        lambda r: sorted([
            (str(r["variable_name_1"]), str(r["definition_1"])),
            (str(r["variable_name_2"]), str(r["definition_2"]))
        ]),
        axis=1
    )

    # add a pair1 and pair2 for filtering later
    df[["pair1", "pair2"]] = pd.DataFrame(
        pairs.tolist(), index=df.index
    )

    # drop dups
    df = (
        df.drop_duplicates(subset=["concept", "pair1", "pair2"])
          .drop(columns=["pair1", "pair2"])
    )
    return df

In [43]:
unique_name_def_df = remove_redundancies(unique_pairs_nodup_filtered)

/tmp/ipykernel_2051119/1824751843.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[["pair1", "pair2"]] = pd.DataFrame(
/tmp/ipykernel_2051119/1824751843.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[["pair1", "pair2"]] = pd.DataFrame(


In [44]:
unique_name_def_df.shape

(922608, 7)

### Get values for each concept
 - we will comb "values" from the thesaurus file
 - values are defined as child concepts for a parent concept

In [45]:
thesaurus_df = pd.read_csv('Thesaurus_26.06e.txt', sep='\t')

In [46]:
thesaurus_df.head(n=2)

,code,conceptIRI,parents,synonyms,definition,displayname,conceptStatus,semantic_type,concept_in_subset
0,C100000,<http://ncicb.nci.nih.gov/xml/owl/EVS/Thesauru...,C99521,Percutaneous Coronary Intervention for ST Elev...,A percutaneous coronary intervention is necess...,NaN,NaN,Therapeutic or Preventive Procedure,CDISC SDTM Cardiac Procedure Indication Termin...
1,C100001,<http://ncicb.nci.nih.gov/xml/owl/EVS/Thesauru...,C99521,Percutaneous Coronary Intervention for ST Elev...,A percutaneous coronary intervention is necess...,NaN,NaN,Therapeutic or Preventive Procedure,CDISC SDTM Cardiac Procedure Indication Termin...


### Values = child concepts for a given code
- to get child concepts, for each code,
- look for rows where code == 'parents'. All concepts in the returned rows are child concepts
- we can then look for the concepts in the synonyms df and capture the variable names as child values

#### example, lets look at Race
- look for rows where parents == C17049
- all rows returned are child concepts or "values" or C17049

In [47]:
thesaurus_df[thesaurus_df['parents'] == 'C17049']

,code,conceptIRI,parents,synonyms,definition,displayname,conceptStatus,semantic_type,concept_in_subset
4942,C104495,<http://ncicb.nci.nih.gov/xml/owl/EVS/Thesauru...,C17049,Other Race|OTHER RACE|RACEOTH,Individuals who do not necessarily identify wi...,NaN,NaN,Population Group,CDISC Variable Terminology|Operational Ontolog...
29064,C126531,<http://ncicb.nci.nih.gov/xml/owl/EVS/Thesauru...,C17049,Latin American|LATIN AMERICAN,Denotes a person whose ancestry is in any of t...,NaN,NaN,Population Group,CDISC CDASH Terminology|CDISC SDTM Collected E...
29068,C126535,<http://ncicb.nci.nih.gov/xml/owl/EVS/Thesauru...,C17049,Australian,Denotes a person whose ancestry is in the coun...,NaN,NaN,Population Group,NaN
29069,C126536,<http://ncicb.nci.nih.gov/xml/owl/EVS/Thesauru...,C17049,New Zealander,Denotes a person whose ancestry is in the coun...,NaN,NaN,Population Group,NaN
64429,C16310,<http://ncicb.nci.nih.gov/xml/owl/EVS/Thesauru...,C17049,Asian American|ASIAN AMERICAN|Asian Americans,Denotes a person having origins in any of the ...,NaN,NaN,Population Group,CDISC CDASH Terminology|CDISC SDTM Collected R...
64618,C16352,<http://ncicb.nci.nih.gov/xml/owl/EVS/Thesauru...,C17049,Black or African American|BLACK OR AFRICAN AME...,Individuals with origins in any of the Black r...,NaN,NaN,Population Group,ALL Authorized Value Terminology|ALL Demograph...
147642,C41219,<http://ncicb.nci.nih.gov/xml/owl/EVS/Thesauru...,C17049,Native Hawaiian or Other Pacific Islander|NATI...,Individuals with origins in any of the origina...,NaN,NaN,Population Group,ALL Authorized Value Terminology|ALL Demograph...
147679,C41260,<http://ncicb.nci.nih.gov/xml/owl/EVS/Thesauru...,C17049,Asian|ASIAN|ASIAN|ASIAN|Asian Ancestry|Asians|...,Individuals with origins in any of the origina...,NaN,NaN,Population Group,ALL Authorized Value Terminology|ALL Demograph...
147680,C41261,<http://ncicb.nci.nih.gov/xml/owl/EVS/Thesauru...,C17049,White|Caucasian|Caucasians|Caucasoid|Occidenta...,Individuals with origins in any of the origina...,NaN,NaN,Population Group,ALL Authorized Value Terminology|ALL Demograph...
148754,C42331,<http://ncicb.nci.nih.gov/xml/owl/EVS/Thesauru...,C17049,African|AFRICAN|African Ancestry|African Descent,Denotes a person with ancestral origins are in...,NaN,NaN,Population Group,CDISC CDASH Terminology|CDISC SDTM Collected R...


In [48]:
thesaurus_df[thesaurus_df['parents'] == 'C17049'].shape

(16, 9)

In [49]:
parent_child_df = thesaurus_df.groupby('parents')['code'].agg(list)

In [50]:
parent_child_df.head(n=2)

parents
C100021    [C100040, C100041, C100042, C100043]
C100032             [C100037, C100038, C100039]
Name: code, dtype: object

In [51]:
parent_child_df.loc['C17049']

['C104495',
 'C126531',
 'C126535',
 'C126536',
 'C16310',
 'C16352',
 'C41219',
 'C41260',
 'C41261',
 'C42331',
 'C43390',
 'C43851',
 'C43866',
 'C67109',
 'C77811',
 'C77812']

#### synonyms df has all child values, i.e variable_names for each code
- e.g. C104495 from the list for race above

In [52]:
synonyms_df.loc['C104495', ]

,name,term_type,source
concept,,,
C104495,OTHER RACE,PT,CDC
C104495,RACEOTH,PT,CDISC
C104495,Other Race,PT,NCI
C104495,Other Race,NaN,NaN
C104495,Other Race,PT,OORO
C104495,Other Race,PT,SeroNet


In [53]:
def remove_value_redundancies(df):
    pairs = df.apply(
        lambda r: tuple(sorted([r["value_list_1"], r["value_list_2"]])),
        axis=1
    )
    df["pair"] = pairs
    result = (
        df.drop_duplicates(subset=["parent_concept_code", "pair"])
          .drop(columns="pair")
    )
    return result
    

In [54]:
def build_positional_value_lists(groups):
    if not groups:
        return []
    max_len = max(len(g) for g in groups)
    positional_lists = []
    for pos in range(max_len):
        col = []
        for group in groups:
            col.append(group[pos] if pos < len(group) else group[-1])
        positional_lists.append(col)
    return positional_lists
 

In [55]:
def to_quoted_csv_list(values):
    return "; ".join(str(v).strip() for v in values if str(v).strip())

def pair_up_lists(positional_lists):
    # we want only value_list_1 and value_list_2 in each row
    # for even number of lists we can split
    # if odd, we will drop
    pairs = []
    for i in range(0, len(positional_lists) - 1, 2):
        pairs.append((positional_lists[i], positional_lists[i + 1]))
    return pairs

def positional_lists_to_dataframe(positional_lists, parent_concept):
    pairs = pair_up_lists(positional_lists)
    rows = [
        {"parent_concept_code": parent_concept, "value_list_1": to_quoted_csv_list(col1), "value_list_2": to_quoted_csv_list(col2)}
        for col1, col2 in pairs
    ]
    return pd.DataFrame(rows)

In [56]:
def create_values_df(parent_concept):
    # get all child codes
    codes = parent_child_df.loc[parent_concept]
    groups = []
    for code in codes:
        vals = list(set(synonyms_df.loc[code]['name'].values))
        groups.append(vals)
    positional_lists = build_positional_value_lists(groups)
    df = positional_lists_to_dataframe(positional_lists, parent_concept)
    return df

In [57]:
# test
create_values_df('C17049')

,parent_concept_code,value_list_1,value_list_2
0,C17049,Other Race; LATIN AMERICAN; Australian; New Ze...,RACEOTH; Latin American; Australian; New Zeala...
1,C17049,OTHER RACE; Latin American; Australian; New Ze...,OTHER RACE; Latin American; Australian; New Ze...
2,C17049,OTHER RACE; Latin American; Australian; New Ze...,OTHER RACE; Latin American; Australian; New Ze...
3,C17049,OTHER RACE; Latin American; Australian; New Ze...,OTHER RACE; Latin American; Australian; New Ze...


In [58]:
parent_child_df.head(n=3)

parents
C100021    [C100040, C100041, C100042, C100043]
C100032             [C100037, C100038, C100039]
C100044                               [C217468]
Name: code, dtype: object

In [59]:
for parent_concept in list(parent_child_df.index):
    # print('processing:', parent_concept)
    if '|' in parent_concept:
        processed_parent_concept = parent_concept.replace("|", "_")
    else:
        processed_parent_concept = parent_concept
    values_output_file = '/'.join(['values_outputs', 
                                   processed_parent_concept + '.values.csv'
                                  ])
    try:
        create_values_df(parent_concept).to_csv(values_output_file)
    except Exception as e:
        pass

In [60]:
print('completed values processing')

completed values processing


In [61]:
unique_name_def_df.head(n=3)

,concept,definition_1,source_1,variable_name_1,definition_2,source_2,variable_name_2
3,C100000,A percutaneous coronary intervention is necess...,CDISC,PERCUTANEOUS CORONARY INTERVENTION (PCI) FOR S...,A percutaneous coronary intervention is necess...,NCI,Percutaneous Coronary Intervention for ST Elev...
6,C100000,A percutaneous coronary intervention is necess...,CDISC,Percutaneous Coronary Intervention for ST Elev...,A percutaneous coronary intervention is necess...,NCI,PERCUTANEOUS CORONARY INTERVENTION (PCI) FOR S...
18,C100001,A percutaneous coronary intervention is necess...,CDISC,PERCUTANEOUS CORONARY INTERVENTION (PCI) FOR S...,A percutaneous coronary intervention is necess...,NCI,Percutaneous Coronary Intervention for ST Elev...


### Merge value lists with `unique_pairs_nodup_filtered`

In [62]:
def get_values_df(concept_name):
    if '|' in concept_name:
        processed_concept_name = concept_name.replace("|", "_")
    else:
        processed_concept_name = concept_name
    try:
        values_df = pd.read_csv('/'.join([
            'values_outputs', processed_concept_name + '.values.csv'
        ]))
    except Exception as e:
        values_df = pd.DataFrame()
    return values_df  

In [63]:
values_df = pd.concat([
    get_values_df(c)
    for c in unique_pairs_nodup_filtered['concept']
])

In [64]:
values_df.shape

(736056, 4)

In [65]:
values_df = values_df.set_index('parent_concept_code')

In [66]:
values_df.head()

,Unnamed: 0,value_list_1,value_list_2
parent_concept_code,,,
C100044,0,Transvenous Pulmonary Valve Balloon Commissuro...,Pulmonary valve commissurotomy by transvenous ...
C100044,0,Transvenous Pulmonary Valve Balloon Commissuro...,Pulmonary valve commissurotomy by transvenous ...
C100047,0,Secure File Transfer Protocol,sFTP
C100047,0,Secure File Transfer Protocol,sFTP
C100047,0,Secure File Transfer Protocol,sFTP


In [67]:
values_df = values_df.drop(columns=["Unnamed: 0"])

In [68]:
values_df = values_df.drop_duplicates()

In [69]:
values_df.shape

(8270, 2)

In [70]:
values_df.head()

,value_list_1,value_list_2
parent_concept_code,,
C100044,Transvenous Pulmonary Valve Balloon Commissuro...,Pulmonary valve commissurotomy by transvenous ...
C100047,Secure File Transfer Protocol,sFTP
C100068,cryoablation; Epicardial Ablation; FIRM Ablati...,Cryoablation for Arrhythmia; Epicardial Ablati...
C100076,Normal Sinus Rhythm,NORMAL SINUS RHYTHM
C100085,Quantitative Coronary Angiography,QUANTITATIVE CORONARY ANGIOGRAPHY


In [71]:
# test
values_df.loc['C17049',]

,value_list_1,value_list_2
parent_concept_code,,
C17049,Other Race; LATIN AMERICAN; Australian; New Ze...,RACEOTH; Latin American; Australian; New Zeala...
C17049,OTHER RACE; Latin American; Australian; New Ze...,OTHER RACE; Latin American; Australian; New Ze...
C17049,OTHER RACE; Latin American; Australian; New Ze...,OTHER RACE; Latin American; Australian; New Ze...
C17049,OTHER RACE; Latin American; Australian; New Ze...,OTHER RACE; Latin American; Australian; New Ze...


In [72]:
unique_name_def_df[unique_name_def_df['concept'] == 'C17049']

,concept,definition_1,source_1,variable_name_1,definition_2,source_2,variable_name_2
2095601,C17049,A geographic ancestral origin category that is...,NCI,Race,An arbitrary classification of a taxonomic gro...,CDISC-GLOSS,race
2095602,C17049,A geographic ancestral origin category that is...,NCI,Race,An arbitrary classification of a taxonomic gro...,CDISC-GLOSS,RACE
2095609,C17049,A geographic ancestral origin category that is...,NCI,Race,An arbitrary classification of a taxonomic gro...,CDISC-GLOSS,Racial Group
2095612,C17049,A geographic ancestral origin category that is...,NCI,Race,An arbitrary classification of a taxonomic gro...,CDISC-GLOSS,Patient Reported Race
2095622,C17049,A geographic ancestral origin category that is...,NCI,Race,An arbitrary classification of a taxonomic gro...,CDISC,race
...,...,...,...,...,...,...,...
2098592,C17049,An arbitrary classification of a taxonomic gro...,CDISC,Racial Group,Race reported by the patient. This is importan...,OORO,Patient Reported Race
2098666,C17049,An arbitrary classification of a taxonomic gro...,CDISC,Patient Reported Race,Race reported by the patient. This is importan...,OORO,Race
2098668,C17049,An arbitrary classification of a taxonomic gro...,CDISC,Patient Reported Race,Race reported by the patient. This is importan...,OORO,race
2098669,C17049,An arbitrary classification of a taxonomic gro...,CDISC,Patient Reported Race,Race reported by the patient. This is importan...,OORO,RACE


In [73]:
values_df = values_df.reset_index()

In [74]:
values_df.head(n=3)

,parent_concept_code,value_list_1,value_list_2
0,C100044,Transvenous Pulmonary Valve Balloon Commissuro...,Pulmonary valve commissurotomy by transvenous ...
1,C100047,Secure File Transfer Protocol,sFTP
2,C100068,cryoablation; Epicardial Ablation; FIRM Ablati...,Cryoablation for Arrhythmia; Epicardial Ablati...


In [75]:
unique_name_def_df_with_values = pd.merge(
    left=unique_name_def_df,
    right=values_df,
    left_on='concept',
    right_on='parent_concept_code',
    how='left'
)

In [76]:
unique_name_def_df.shape

(922608, 7)

In [77]:
values_df.shape

(8270, 3)

In [78]:
# only 4166 variables with semantically interop values
values_df['parent_concept_code'].nunique()

4166

In [79]:
unique_name_def_df_with_values.shape

(1370164, 10)

In [80]:
unique_name_def_df_with_values[unique_name_def_df_with_values['concept'] == 'C17049']

,concept,definition_1,source_1,variable_name_1,definition_2,source_2,variable_name_2,parent_concept_code,value_list_1,value_list_2
390999,C17049,A geographic ancestral origin category that is...,NCI,Race,An arbitrary classification of a taxonomic gro...,CDISC-GLOSS,race,C17049,Other Race; LATIN AMERICAN; Australian; New Ze...,RACEOTH; Latin American; Australian; New Zeala...
391000,C17049,A geographic ancestral origin category that is...,NCI,Race,An arbitrary classification of a taxonomic gro...,CDISC-GLOSS,race,C17049,OTHER RACE; Latin American; Australian; New Ze...,OTHER RACE; Latin American; Australian; New Ze...
391001,C17049,A geographic ancestral origin category that is...,NCI,Race,An arbitrary classification of a taxonomic gro...,CDISC-GLOSS,race,C17049,OTHER RACE; Latin American; Australian; New Ze...,OTHER RACE; Latin American; Australian; New Ze...
391002,C17049,A geographic ancestral origin category that is...,NCI,Race,An arbitrary classification of a taxonomic gro...,CDISC-GLOSS,race,C17049,OTHER RACE; Latin American; Australian; New Ze...,OTHER RACE; Latin American; Australian; New Ze...
391003,C17049,A geographic ancestral origin category that is...,NCI,Race,An arbitrary classification of a taxonomic gro...,CDISC-GLOSS,RACE,C17049,Other Race; LATIN AMERICAN; Australian; New Ze...,RACEOTH; Latin American; Australian; New Zeala...
...,...,...,...,...,...,...,...,...,...,...
391474,C17049,An arbitrary classification of a taxonomic gro...,CDISC,Patient Reported Race,Race reported by the patient. This is importan...,OORO,RACE,C17049,OTHER RACE; Latin American; Australian; New Ze...,OTHER RACE; Latin American; Australian; New Ze...
391475,C17049,An arbitrary classification of a taxonomic gro...,CDISC,Patient Reported Race,Race reported by the patient. This is importan...,OORO,Racial Group,C17049,Other Race; LATIN AMERICAN; Australian; New Ze...,RACEOTH; Latin American; Australian; New Zeala...
391476,C17049,An arbitrary classification of a taxonomic gro...,CDISC,Patient Reported Race,Race reported by the patient. This is importan...,OORO,Racial Group,C17049,OTHER RACE; Latin American; Australian; New Ze...,OTHER RACE; Latin American; Australian; New Ze...
391477,C17049,An arbitrary classification of a taxonomic gro...,CDISC,Patient Reported Race,Race reported by the patient. This is importan...,OORO,Racial Group,C17049,OTHER RACE; Latin American; Australian; New Ze...,OTHER RACE; Latin American; Australian; New Ze...


In [81]:
unique_name_def_df_with_values.columns

Index(['concept', 'definition_1', 'source_1', 'variable_name_1',
       'definition_2', 'source_2', 'variable_name_2', 'parent_concept_code',
       'value_list_1', 'value_list_2'],
      dtype='object')

In [82]:
unique_name_def_df_with_values = unique_name_def_df_with_values.rename(columns={
    'definition_1': 'variable_description_1',
    'definition_2': 'variable_description_2',
    'value_list_1': 'variable_value_list_1',
    'value_list_2': 'variable_value_list_2'
})

In [83]:
unique_name_def_df_with_values.head(n=3)

,concept,variable_description_1,source_1,variable_name_1,variable_description_2,source_2,variable_name_2,parent_concept_code,variable_value_list_1,variable_value_list_2
0,C100000,A percutaneous coronary intervention is necess...,CDISC,PERCUTANEOUS CORONARY INTERVENTION (PCI) FOR S...,A percutaneous coronary intervention is necess...,NCI,Percutaneous Coronary Intervention for ST Elev...,NaN,NaN,NaN
1,C100000,A percutaneous coronary intervention is necess...,CDISC,Percutaneous Coronary Intervention for ST Elev...,A percutaneous coronary intervention is necess...,NCI,PERCUTANEOUS CORONARY INTERVENTION (PCI) FOR S...,NaN,NaN,NaN
2,C100001,A percutaneous coronary intervention is necess...,CDISC,PERCUTANEOUS CORONARY INTERVENTION (PCI) FOR S...,A percutaneous coronary intervention is necess...,NCI,Percutaneous Coronary Intervention for ST Elev...,NaN,NaN,NaN


In [84]:
unique_name_def_df_with_values.columns

Index(['concept', 'variable_description_1', 'source_1', 'variable_name_1',
       'variable_description_2', 'source_2', 'variable_name_2',
       'parent_concept_code', 'variable_value_list_1',
       'variable_value_list_2'],
      dtype='object')

In [85]:
unique_name_def_df_with_values["variable_para"] = unique_name_def_df_with_values.apply(
    lambda x: " | ".join([
        "name: "        + (str(x["variable_name_1"])        if pd.notna(x["variable_name_1"])        else ""),
        "description: " + (str(x["variable_description_1"]) if pd.notna(x["variable_description_1"]) else ""),
        "values: "      + (str(x["variable_value_list_1"])  if pd.notna(x["variable_value_list_1"])  else ""),
    ]).strip(),
    axis=1
)

In [86]:
unique_name_def_df_with_values["alternate_variable_para"] = unique_name_def_df_with_values.apply(
    lambda x: " | ".join([
        "name: "        + (str(x["variable_name_2"])        if pd.notna(x["variable_name_2"])        else ""),
        "description: " + (str(x["variable_description_2"]) if pd.notna(x["variable_description_2"]) else ""),
        "values: "      + (str(x["variable_value_list_2"])  if pd.notna(x["variable_value_list_2"])  else ""),
    ]).strip(),
    axis=1
)

In [87]:
# check
unique_name_def_df_with_values[unique_name_def_df_with_values['concept'] == 'C17049'].to_csv('test.csv')

In [88]:
# output to file
unique_name_def_df_with_values.to_csv('ncit_training.csv')

In [89]:
### NOTE: definitions from NCIt may already be very similar, we havent filtered them out